In [2]:
!pip install ucimlrepo


In [4]:
# 🚀 Install dependencies (run once)
# pip install ucimlrepo scikit-learn torch pandas

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from ucimlrepo import fetch_ucirepo

# 1️⃣ Load dataset from UCI
credit = fetch_ucirepo(id=144)  # German Credit Dataset
X = credit.data.features
y = credit.data.targets

# Convert to DataFrame for easier handling
X = pd.DataFrame(X)
y = pd.Series(y.values.flatten(), name="target")

# 2️⃣ Encode categorical columns (label encoding)
for col in X.columns:
    if X[col].dtype == 'object':
        X[col] = LabelEncoder().fit_transform(X[col])

# 3️⃣ Split into train/test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4️⃣ Normalize numerical features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert to PyTorch tensors and adjust target values to be 0 or 1
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train.values - 1, dtype=torch.float32).view(-1, 1)
y_test = torch.tensor(y_test.values - 1, dtype=torch.float32).view(-1, 1)

# 5️⃣ Define the MLP model
class MLPModel(nn.Module):
    def __init__(self, input_dim):
        super(MLPModel, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, 1),
            nn.Sigmoid()  # For binary classification
        )

    def forward(self, x):
        return self.net(x)

model = MLPModel(input_dim=X_train.shape[1])

# 6️⃣ Define loss & optimizer
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

# 7️⃣ Train the model
epochs = 50
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()
    outputs = model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# 8️⃣ Evaluate
model.eval()
with torch.no_grad():
    preds = model(X_test)
    predicted = (preds > 0.5).float()
    accuracy = (predicted.eq(y_test).sum() / y_test.shape[0]).item()

print(f"\n✅ Test Accuracy: {accuracy*100:.2f}%")

Epoch [10/50], Loss: 0.5221
Epoch [20/50], Loss: 0.4591
Epoch [30/50], Loss: 0.4230
Epoch [40/50], Loss: 0.3849
Epoch [50/50], Loss: 0.3355

✅ Test Accuracy: 75.50%
